In [30]:
import os
import json
import hashlib

import cv2
import numpy as np
import pandas as pd
import mediapipe as mp

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader


In [31]:
# Main dataset folder
DATASET_PATH = r"D:\sign language\video sentence\paul"

# CSV will be created automatically
CSV_PATH = r"D:\sign language\video sentence\paul\dataset.csv"

# Output folder
OUTPUT_PATH = r"D:\sign language\output"

# MediaPipe cache
CACHE_PATH = r"D:\sign language\output\cache"


# Create output folders
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CACHE_PATH, exist_ok=True)



In [32]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Using: cpu


In [33]:
# Maximum number of frames for Transformer
MAX_FRAMES = 64

# Maximum number of hands
NUM_HANDS = 2

# MediaPipe hand landmarks
LANDMARKS_PER_HAND = 21

# x, y, z
COORDINATES = 3

# 2 hands × 21 landmarks × 3 coordinates
INPUT_SIZE = (
    NUM_HANDS
    * LANDMARKS_PER_HAND
    * COORDINATES
)

# Training
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 0.0001

# Validation percentage
VALIDATION_SIZE = 0.20

# Random seed
RANDOM_STATE = 42


In [34]:
if not os.path.exists(DATASET_PATH):

    raise FileNotFoundError(
        f"\nDataset folder not found:\n{DATASET_PATH}"
    )


print("\nDataset path:")
print(DATASET_PATH)



Dataset path:
D:\sign language\video sentence\paul


In [35]:
def create_dataset_csv():
    records = []
    word_path = os.path.join(DATASET_PATH,"word")

    sentence_path = os.path.join(DATASET_PATH,"sentence")

    if os.path.exists(word_path):

        print("\nScanning WORD dataset...")

        for label in sorted(os.listdir(word_path)):

            label_path = os.path.join(word_path,label)

            if not os.path.isdir(label_path):
                continue


            for file in sorted(os.listdir(label_path)):

                if file.lower().endswith((".jpg", ".jpeg", ".png")):

                    image_path = os.path.join(label_path,file)

                    records.append({
                        "path": image_path,
                        "label": label,
                        "type": "word"
                    })

    if os.path.exists(sentence_path):

        print("\nScanning SENTENCE dataset...")

        for label in sorted(os.listdir(sentence_path)):

            label_path = os.path.join(sentence_path,label)

            if not os.path.isdir(label_path):
                continue


            for sample in sorted(os.listdir(label_path)):

                sample_path = os.path.join(label_path,sample)

                if not os.path.isdir(sample_path):
                    continue


                # Check if folder contains images

                images = []

                for file in os.listdir(sample_path):
                    if file.lower().endswith((".jpg", ".jpeg", ".png")):
                        images.append(file)


                if len(images) == 0:
                    continue


                records.append({
                    "path": sample_path,
                    "label": label,
                    "type": "sentence"
                })


    df = pd.DataFrame(records)


    if len(df) == 0:

        raise ValueError(
            "\nNo images or frame folders were found."
            "\nCheck DATASET_PATH."
        )


    # Save CSV

    df.to_csv(
        CSV_PATH,
        index=False,
        encoding="utf-8"
    )


    return df


# ============================================================
# CREATE CSV
# ============================================================

print("\n" + "=" * 70)
print("CREATING DATASET CSV")
print("=" * 70)

df = create_dataset_csv()



CREATING DATASET CSV

Scanning WORD dataset...

Scanning SENTENCE dataset...


In [36]:
print(
    "Total labels:",
    df["label"].nunique()
)


Total labels: 6


In [37]:
print(df.shape)

print(df.columns.tolist())

print(df["type"].value_counts())

print(df["label"].value_counts().head(20))

print(df["label"].value_counts().min())

print(df["label"].value_counts().max())

(45, 3)
['path', 'label', 'type']
type
word        24
sentence    21
Name: count, dtype: int64
label
DO                    12
BED                    7
are_you_free_today     7
do_not_worry           7
help_me                7
ABUSE                  5
Name: count, dtype: int64
5
12


In [38]:

labels = sorted(df["label"].unique())


label_to_id = {label: index 
               for index, label in enumerate(labels)}


id_to_label = { index: label 
               for label, index in label_to_id.items()}


df["label_id"] = df["label"].map(label_to_id)


NUM_CLASSES = len(labels)


In [39]:
with open(
    os.path.join(OUTPUT_PATH,"labels.json"),
    "w",
    encoding="utf-8" ) as file:

    json.dump(
        label_to_id,
        file,
        indent=4,
        ensure_ascii=False
    )



In [40]:
import mediapipe as mp

print("MediaPipe:", mp.__version__)
print("Solutions available:", hasattr(mp, "solutions"))

mp_hands = mp.solutions.hands

print("Hands available:", mp_hands)

MediaPipe: 0.10.21
Solutions available: True
Hands available: <module 'mediapipe.python.solutions.hands' from 'd:\\sign language\\.venv\\Lib\\site-packages\\mediapipe\\python\\solutions\\hands.py'>


In [41]:
mp_hands = mp.solutions.hands


hands = mp_hands.Hands(

    static_image_mode=False,

    max_num_hands=NUM_HANDS,

    min_detection_confidence=0.5,

    min_tracking_confidence=0.5
)



In [42]:

def get_cache_path(sample_path):

    # Create stable hash

    hash_name = hashlib.md5(sample_path.encode("utf-8")).hexdigest()


    return os.path.join(CACHE_PATH,hash_name + ".npz")


# ============================================================
# 11. NORMALIZE HAND LANDMARKS
# ============================================================

def normalize_landmarks(hand_landmarks):

    points = []

    for landmark in hand_landmarks.landmark:

        points.append([
            landmark.x,
            landmark.y,
            landmark.z
        ])


    points = np.array( points , dtype=np.float32)


    # --------------------------------------------------------
    # Make wrist the origin
    # --------------------------------------------------------

    wrist = points[0].copy()

    points = points - wrist


    # --------------------------------------------------------
    # Scale normalization
    # --------------------------------------------------------

    distances = np.linalg.norm( points , axis=1)


    scale = np.max(distances)


    if scale > 0:

        points = ( points / scale )


    return points.flatten()


In [43]:

def process_image(image_path):

    image = cv2.imread(image_path)


    if image is None:

        return np.zeros(
            INPUT_SIZE,
            dtype=np.float32
        )


    rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )


    result = hands.process(rgb)


    frame_features = np.zeros(
        INPUT_SIZE,
        dtype=np.float32
    )


    if not result.multi_hand_landmarks:

        return frame_features


  
    # Try to keep left/right hand ordering stable
  

    hands_data = []


    if result.multi_hand_landmarks:

        for index, hand_landmarks in enumerate(result.multi_hand_landmarks):

            handedness = None

            if (
                result.multi_handedness
                and index < len(result.multi_handedness)
            ):

                handedness = (
                    result
                    .multi_handedness[index]
                    .classification[0]
                    .label
                )


            landmarks = normalize_landmarks(
                hand_landmarks
            )


            hands_data.append((handedness,landmarks))


  
    # Left hand -> position 0
    # Right hand -> position 1
  

    left_hand = None
    right_hand = None
    unknown_hands = []


    for handedness, landmarks in hands_data:

        if handedness == "Left":

            left_hand = landmarks

        elif handedness == "Right":

            right_hand = landmarks

        else:

            unknown_hands.append(
                landmarks
            )


    # Fill missing hands
    if left_hand is None and unknown_hands:

        left_hand = unknown_hands.pop(0)


    if right_hand is None and unknown_hands:

        right_hand = unknown_hands.pop(0)


  
    # Put landmarks into 126 feature vector
  

    if left_hand is not None:

        frame_features[0:63] = left_hand


    if right_hand is not None:

        frame_features[63:126] = right_hand


    return frame_features



In [44]:

def get_images_from_sample(
    sample_path,
    sample_type
):

    # --------------------------------------------------------
    # WORD
    #
    # path = BED/1.jpg
    # --------------------------------------------------------

    if sample_type == "word":

        return [
            sample_path
        ]


    # --------------------------------------------------------
    # SENTENCE
    #
    # path = do_not_worry/2/
    # --------------------------------------------------------

    images = []


    if os.path.isdir(
        sample_path
    ):

        for file in os.listdir(
            sample_path
        ):

            if file.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):

                images.append(
                    os.path.join(
                        sample_path,
                        file
                    )
                )


    # Sort frame names numerically when possible

    def natural_sort_key(path):

        name = os.path.splitext(
            os.path.basename(path)
        )[0]

        try:

            return int(name)

        except:

            return name


    images.sort(
        key=natural_sort_key
    )


    return images


In [45]:
# 14. EXTRACT FEATURES FROM ONE SAMPLE
# ============================================================

def extract_sample_features(
    sample_path,
    sample_type
):

    cache_file = get_cache_path(
        sample_path
    )


    # --------------------------------------------------------
    # LOAD CACHE
    # --------------------------------------------------------

    if os.path.exists(
        cache_file
    ):

        cached = np.load(
            cache_file
        )

        return (
            cached["features"],
            cached["mask"]
        )


    # --------------------------------------------------------
    # GET IMAGES
    # --------------------------------------------------------

    image_paths = get_images_from_sample(
        sample_path,
        sample_type
    )


    if len(image_paths) == 0:

        return None, None


    # --------------------------------------------------------
    # Select MAX_FRAMES
    # --------------------------------------------------------

    if len(image_paths) > MAX_FRAMES:

        indexes = np.linspace(
            0,
            len(image_paths) - 1,
            MAX_FRAMES
        ).astype(int)

        image_paths = [
            image_paths[i]
            for i in indexes
        ]


    # --------------------------------------------------------
    # Process frames
    # --------------------------------------------------------

    features = []


    for image_path in image_paths:

        feature = process_image(
            image_path
        )

        features.append(
            feature
        )


    features = np.array(
        features,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Valid-frame mask
    # --------------------------------------------------------

    valid_frames = len(
        features
    )


    mask = np.zeros(
        MAX_FRAMES,
        dtype=bool
    )


    mask[
        :valid_frames
    ] = True


    # --------------------------------------------------------
    # Padding
    # --------------------------------------------------------

    if valid_frames < MAX_FRAMES:

        padding = np.zeros(
            (
                MAX_FRAMES - valid_frames,
                INPUT_SIZE
            ),
            dtype=np.float32
        )


        features = np.vstack([
            features,
            padding
        ])


    else:

        features = features[
            :MAX_FRAMES
        ]


    # --------------------------------------------------------
    # Save cache
    # --------------------------------------------------------

    np.savez_compressed(
        cache_file,
        features=features,
        mask=mask
    )


    return features, mask


In [46]:
# 15. EXTRACT WHOLE DATASET
# ============================================================

print("\n" + "=" * 70)
print("EXTRACTING MEDIAPIPE FEATURES")
print("=" * 70)


X = []
y = []
masks = []

failed = 0


for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Processing"
):

    features, mask = extract_sample_features(

        row["path"],

        row["type"]
    )


    if features is None:

        failed += 1

        continue


    X.append(
        features
    )

    masks.append(
        mask
    )

    y.append(
        row["label_id"]
    )


# Convert to NumPy

X = np.array(
    X,
    dtype=np.float32
)


y = np.array(
    y,
    dtype=np.int64
)


masks = np.array(
    masks,
    dtype=bool
)


print("\nFeature shape:")
print(X.shape)


print("\nMask shape:")
print(masks.shape)


print("\nLabel shape:")
print(y.shape)


print("\nFailed samples:")
print(failed)


# ============================================================
# 16. CHECK DATA
# ============================================================

if len(X) == 0:

    raise ValueError(
        "\nNo features were generated."
    )


if len(X) != len(y):

    raise ValueError(
        "\nX and y sizes do not match."
    )




EXTRACTING MEDIAPIPE FEATURES


Processing: 100%|██████████| 45/45 [00:57<00:00,  1.28s/it]


Feature shape:
(45, 64, 126)

Mask shape:
(45, 64)

Label shape:
(45,)

Failed samples:
0


In [47]:
# 17. SAVE FEATURES
# ============================================================

print("\nSaving extracted features...")


np.save(
    os.path.join(
        OUTPUT_PATH,
        "X_features.npy"
    ),
    X
)


np.save(
    os.path.join(
        OUTPUT_PATH,
        "y_labels.npy"
    ),
    y
)


np.save(
    os.path.join(
        OUTPUT_PATH,
        "frame_masks.npy"
    ),
    masks
)



Saving extracted features...


In [48]:

class_counts = np.bincount(
    y
)


minimum_class_count = class_counts.min()


validation_count = int(
    len(y) * VALIDATION_SIZE
)


can_stratify = (

    minimum_class_count >= 2

    and validation_count >= NUM_CLASSES

)


if can_stratify:

    print(
        "Using stratified split."
    )

    (
        X_train,
        X_val,
        y_train,
        y_val,
        mask_train,
        mask_val
    ) = train_test_split(

        X,
        y,
        masks,

        test_size=VALIDATION_SIZE,

        random_state=RANDOM_STATE,

        stratify=y
    )


else:

    print(
        "Stratified split is not possible."
    )

    print(
        "Using normal random split."
    )

    (
        X_train,
        X_val,
        y_train,
        y_val,
        mask_train,
        mask_val
    ) = train_test_split(

        X,
        y,
        masks,

        test_size=VALIDATION_SIZE,

        random_state=RANDOM_STATE
    )


print(
    "\nTraining samples:",
    len(X_train)
)


print(
    "Validation samples:",
    len(X_val)
)


Using stratified split.

Training samples: 36
Validation samples: 9


In [49]:
# 19. PYTORCH DATASET

class SignDataset(
    Dataset
):

    def __init__(
        self,
        X,
        y,
        masks
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

        self.masks = torch.tensor(
            masks,
            dtype=torch.bool
        )


    def __len__(self):

        return len(
            self.X
        )


    def __getitem__(
        self,
        index
    ):

        return (
            self.X[index],
            self.y[index],
            self.masks[index]
        )



In [50]:
# 20. DATA LOADERS

train_dataset = SignDataset(
    X_train,
    y_train,
    mask_train
)


val_dataset = SignDataset(
    X_val,
    y_val,
    mask_val
)


train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0
)


In [53]:
# 21. TRANSFORMER MODEL


class SignTransformer(
    nn.Module
):

    def __init__(
        self,
        num_classes
    ):

        super().__init__()


        # ----------------------------------------------------
        # Input layer
        # 126 -> 256
        # ----------------------------------------------------

        self.input_layer = nn.Sequential(

            nn.Linear(
                INPUT_SIZE,
                256
            ),

            nn.LayerNorm(
                256
            )
        )


        # ----------------------------------------------------
        # Positional embedding
        # ----------------------------------------------------

        self.position = nn.Parameter(

            torch.randn(
                1,
                MAX_FRAMES,
                256
            )
            * 0.02
        )


        # ----------------------------------------------------
        # Transformer Encoder
        # ----------------------------------------------------

        encoder_layer = (

            nn.TransformerEncoderLayer(

                d_model=256,

                nhead=8,

                dim_feedforward=1024,

                dropout=0.1,

                batch_first=True,

                activation="gelu"
            )
        )


        self.transformer = (

            nn.TransformerEncoder(

                encoder_layer,

                num_layers=4
            )
        )


        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.LayerNorm(
                256
            ),

            nn.Linear(
                256,
                256
            ),

            nn.GELU(),

            nn.Dropout(
                0.2
            ),

            nn.Linear(
                256,
                num_classes
            )
        )


    def forward(
        self,
        x,
        mask
    ):

        # ----------------------------------------------------
        # Convert input
        # ----------------------------------------------------

        x = self.input_layer(
            x
        )


        # ----------------------------------------------------
        # Position
        # ----------------------------------------------------

        x = x + self.position[
            :,
            :x.size(1),
            :
        ]


        # ----------------------------------------------------
        # Transformer
        #
        # Transformer expects:
        # True = padding
        # False = valid
        # ----------------------------------------------------

        padding_mask = ~mask


        x = self.transformer(

            x,

            src_key_padding_mask=padding_mask
        )


        # ----------------------------------------------------
        # Masked mean pooling
        # ----------------------------------------------------

        valid_mask = mask.unsqueeze(
            -1
        ).float()


        x = x * valid_mask


        sum_features = x.sum(
            dim=1
        )


        count_features = (
            valid_mask
            .sum(dim=1)
            .clamp(min=1)
        )


        x = (
            sum_features
            / count_features
        )


        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        return self.classifier(
            x
        )




In [54]:
# 22. CREATE MODEL

model = SignTransformer(
    NUM_CLASSES
).to(DEVICE)


print("\n" + "=" * 70)
print("MODEL")
print("=" * 70)


print(
    "Input size:",
    INPUT_SIZE
)


print(
    "Maximum frames:",
    MAX_FRAMES
)


print(
    "Classes:",
    NUM_CLASSES
)


print(
    "Parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)




MODEL
Input size: 126
Maximum frames: 64
Classes: 6
Parameters: 3276294


In [55]:
# 23. LOSS + OPTIMIZER

criterion = nn.CrossEntropyLoss()


optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=0.01
)


# Learning-rate scheduler

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=3
)


In [56]:
# 24. TRAINING

best_accuracy = 0.0


history = []


print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)


for epoch in range(
    EPOCHS
):


    # ========================================================
    # TRAIN
    # ========================================================

    model.train()


    train_loss = 0.0

    train_correct = 0

    train_total = 0


    for (
        batch_X,
        batch_y,
        batch_mask
    ) in train_loader:


        batch_X = batch_X.to(
            DEVICE
        )


        batch_y = batch_y.to(
            DEVICE
        )


        batch_mask = batch_mask.to(
            DEVICE
        )


        # Reset gradients

        optimizer.zero_grad()


        # Forward

        output = model(

            batch_X,

            batch_mask
        )


        # Loss

        loss = criterion(

            output,

            batch_y
        )


        # Backpropagation

        loss.backward()


        # Gradient clipping

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0
        )


        # Update

        optimizer.step()


        # Statistics

        train_loss += (
            loss.item()
        )


        prediction = output.argmax(
            dim=1
        )


        train_correct += (

            prediction == batch_y

        ).sum().item()


        train_total += len(
            batch_y
        )


    # ========================================================
    # TRAIN ACCURACY
    # ========================================================

    train_accuracy = (

        train_correct
        / train_total

    )


    train_loss = (

        train_loss
        / len(train_loader)

    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()


    val_loss = 0.0

    val_correct = 0

    val_total = 0


    all_predictions = []

    all_targets = []


    with torch.no_grad():

        for (
            batch_X,
            batch_y,
            batch_mask
        ) in val_loader:


            batch_X = batch_X.to(
                DEVICE
            )


            batch_y = batch_y.to(
                DEVICE
            )


            batch_mask = batch_mask.to(
                DEVICE
            )


            output = model(

                batch_X,

                batch_mask
            )


            loss = criterion(

                output,

                batch_y
            )


            val_loss += (
                loss.item()
            )


            prediction = output.argmax(
                dim=1
            )


            val_correct += (

                prediction == batch_y

            ).sum().item()


            val_total += len(
                batch_y
            )


            all_predictions.extend(
                prediction.cpu().numpy()
            )


            all_targets.extend(
                batch_y.cpu().numpy()
            )


    # ========================================================
    # VALIDATION STATISTICS
    # ========================================================

    val_accuracy = (

        val_correct
        / val_total

    )


    val_loss = (

        val_loss
        / len(val_loader)

    )


    # Scheduler

    scheduler.step(
        val_accuracy
    )


    current_lr = optimizer.param_groups[0]["lr"]


    # ========================================================
    # PRINT
    # ========================================================

    print("\n" + "-" * 70)

    print(
        f"Epoch "
        f"{epoch + 1}/{EPOCHS}"
    )


    print(
        f"Train Loss: "
        f"{train_loss:.4f}"
    )


    print(
        f"Train Accuracy: "
        f"{train_accuracy * 100:.2f}%"
    )


    print(
        f"Val Loss: "
        f"{val_loss:.4f}"
    )


    print(
        f"Val Accuracy: "
        f"{val_accuracy * 100:.2f}%"
    )


    print(
        f"Learning Rate: "
        f"{current_lr:.8f}"
    )


    # ========================================================
    # HISTORY
    # ========================================================

    history.append({

        "epoch":
            epoch + 1,

        "train_loss":
            train_loss,

        "train_accuracy":
            train_accuracy,

        "val_loss":
            val_loss,

        "val_accuracy":
            val_accuracy,

        "learning_rate":
            current_lr

    })


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy


        checkpoint = {

            "model_state_dict":
                model.state_dict(),

            "num_classes":
                NUM_CLASSES,

            "labels":
                label_to_id,

            "input_size":
                INPUT_SIZE,

            "max_frames":
                MAX_FRAMES,

            "best_accuracy":
                best_accuracy

        }


        torch.save(

            checkpoint,

            os.path.join(

                OUTPUT_PATH,

                "sign_transformer_best.pth"
            )
        )


        print(
            "\n*** BEST MODEL SAVED ***"
        )




TRAINING


d:\sign language\.venv\Lib\site-packages\torch\nn\modules\transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(



----------------------------------------------------------------------
Epoch 1/30
Train Loss: 1.8068
Train Accuracy: 19.44%
Val Loss: 1.6460
Val Accuracy: 11.11%
Learning Rate: 0.00010000

*** BEST MODEL SAVED ***

----------------------------------------------------------------------
Epoch 2/30
Train Loss: 1.5295
Train Accuracy: 55.56%
Val Loss: 1.5929
Val Accuracy: 11.11%
Learning Rate: 0.00010000

----------------------------------------------------------------------
Epoch 3/30
Train Loss: 1.3832
Train Accuracy: 58.33%
Val Loss: 1.5184
Val Accuracy: 11.11%
Learning Rate: 0.00010000

----------------------------------------------------------------------
Epoch 4/30
Train Loss: 1.1471
Train Accuracy: 69.44%
Val Loss: 1.3661
Val Accuracy: 33.33%
Learning Rate: 0.00010000

*** BEST MODEL SAVED ***

----------------------------------------------------------------------
Epoch 5/30
Train Loss: 1.0348
Train Accuracy: 75.00%
Val Loss: 1.2471
Val Accuracy: 33.33%
Learning Rate: 0.00010000

--

In [57]:

torch.save(

    {

        "model_state_dict":
            model.state_dict(),

        "num_classes":
            NUM_CLASSES,

        "labels":
            label_to_id,

        "input_size":
            INPUT_SIZE,

        "max_frames":
            MAX_FRAMES

    },

    os.path.join(

        OUTPUT_PATH,

        "sign_transformer_final.pth"
    )
)


# 26. SAVE TRAINING HISTORY

history_df = pd.DataFrame(
    history
)


history_df.to_csv(

    os.path.join(

        OUTPUT_PATH,

        "training_history.csv"
    ),

    index=False
)



In [ ]:
# 27. TRAINING GRAPH

plt.figure(
    figsize=(10, 5)
)


plt.plot(

    history_df[
        "train_accuracy"
    ] * 100,

    label="Train Accuracy"
)


plt.plot(

    history_df[
        "val_accuracy"
    ] * 100,

    label="Validation Accuracy"
)


plt.xlabel(
    "Epoch"
)


plt.ylabel(
    "Accuracy (%)"
)


plt.title(
    "Training and Validation Accuracy"
)


plt.legend()


plt.tight_layout()


plt.savefig(

    os.path.join(

        OUTPUT_PATH,

        "accuracy.png"
    )
)


plt.close()


# 28. LOSS GRAPH

plt.figure(
    figsize=(10, 5)
)


plt.plot(

    history_df[
        "train_loss"
    ],

    label="Train Loss"
)


plt.plot(

    history_df[
        "val_loss"
    ],

    label="Validation Loss"
)


plt.xlabel(
    "Epoch"
)


plt.ylabel(
    "Loss"
)


plt.title(
    "Training and Validation Loss"
)


plt.legend()


plt.tight_layout()


plt.savefig(

    os.path.join(

        OUTPUT_PATH,

        "loss.png"
    )
)


plt.close()


# 29. CONFUSION MATRIX

print("\nGenerating confusion matrix...")


model.eval()


all_predictions = []

all_targets = []


with torch.no_grad():

    for (
        batch_X,
        batch_y,
        batch_mask
    ) in val_loader:


        batch_X = batch_X.to(
            DEVICE
        )


        batch_mask = batch_mask.to(
            DEVICE
        )


        output = model(

            batch_X,

            batch_mask
        )


        prediction = output.argmax(
            dim=1
        )


        all_predictions.extend(
            prediction.cpu().numpy()
        )


        all_targets.extend(
            batch_y.numpy()
        )


# Only display classes actually appearing
# in validation when there are many classes.

present_classes = sorted(
    set(all_targets)
    | set(all_predictions)
)


cm = confusion_matrix(

    all_targets,

    all_predictions,

    labels=present_classes
)


# For a very large number of classes,
# the matrix can become difficult to read.

if len(present_classes) <= 50:

    plt.figure(
        figsize=(14, 12)
    )


    class_names = [

        id_to_label[
            class_id
        ]

        for class_id in present_classes

    ]


    sns.heatmap(

        cm,

        annot=False,

        fmt="d",

        xticklabels=class_names,

        yticklabels=class_names
    )


    plt.xlabel(
        "Predicted"
    )


    plt.ylabel(
        "Actual"
    )


    plt.title(
        "Confusion Matrix"
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            OUTPUT_PATH,

            "confusion_matrix.png"
        )
    )


    plt.close()


# 30. CLASSIFICATION REPORT

try:

    report = classification_report(

        all_targets,

        all_predictions,

        labels=present_classes,

        target_names=[

            id_to_label[
                i
            ]

            for i in present_classes

        ],

        zero_division=0
    )


    print(
        "\nClassification Report:"
    )

    print(report)


    with open(

        os.path.join(

            OUTPUT_PATH,

            "classification_report.txt"
        ),

        "w",

        encoding="utf-8"

    ) as file:

        file.write(
            report
        )


except Exception as e:

    print(
        "Could not create classification report:",
        e
    )


# 31. FINAL INFORMATION

print("\n")
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)


print(
    "\nBest validation accuracy:"
)


print(
    f"{best_accuracy * 100:.2f}%"
)


print(
    "\nOutput folder:"
)


print(
    OUTPUT_PATH
)





Generating confusion matrix...

Classification Report:
                    precision    recall  f1-score   support

             ABUSE       0.00      0.00      0.00         1
               BED       1.00      0.50      0.67         2
                DO       0.50      0.50      0.50         2
are_you_free_today       0.25      1.00      0.40         1
      do_not_worry       0.00      0.00      0.00         2
           help_me       0.00      0.00      0.00         1

          accuracy                           0.33         9
         macro avg       0.29      0.33      0.26         9
      weighted avg       0.36      0.33      0.30         9



TRAINING COMPLETE

Best validation accuracy:
44.44%

Output folder:
D:\sign language\output

Files created:
1. dataset.csv
2. labels.json
3. X_features.npy
4. y_labels.npy
5. frame_masks.npy
6. sign_transformer_best.pth
7. sign_transformer_final.pth
8. training_history.csv
9. accuracy.png
10. loss.png
11. classification_report.txt
12. co